# 22-05 · Flask: trasy, wzory i kształty

Praktyka do sekcji [„Pierwsza aplikacja webowa na Flask” `projects/flask/todo-app/app.py`.

## O kliencie testowym w tym laptopie

Zazwyczaj aplikacja Flask-uruchamiana jest za pomocą polecenia `python app.py`i czeka na żądania przeglądarki w nieskończoność (`app.run()`). W automatycznie wykonywanym notebooku nie ma przeglądarki i nie ma sensu czekać wiecznie, dlatego używamy `app.test_client()`. Konstruuje żądanie w procesie i przekazuje je aplikacji przez interfejs testowy Flask/Werkzeug, bez otwierania portu sieciowego lub uruchamiania prawdziwego serwera — ta sama logika przetwarzania żądań co w rzeczywistej aplikacji, ale bez sieci (sekcja 22.33 strony omawia tę różnicę bardziej szczegółowo).

## Praktyczny przykład — składamy małą aplikację

In [ ]:
from flask import Flask, redirect, render_template_string, request, url_for

app = Flask(__name__)
zadachi = ["Выучить основы Python", "Собрать сайт на Flask"]

SHABLON_GLAVNOJ = """
<h1>Мой список задач</h1>
<ul>
{% for zadacha in zadachi %}
  <li>{{ zadacha }}</li>
{% endfor %}
</ul>
"""

SHABLON_PRIVET = "<h1>Привет, {{ imya }}!</h1>"


@app.route("/")
def glavnaya():
    return render_template_string(SHABLON_GLAVNOJ, zadachi=zadachi)


@app.route("/privet/<imya>")
def privet(imya):
    return render_template_string(SHABLON_PRIVET, imya=imya)


@app.route("/dobavit", methods=["POST"])
def dobavit():
    novaya_zadacha = request.form.get("zadacha", "").strip()
    if novaya_zadacha:
        zadachi.append(novaya_zadacha)
    return redirect(url_for("glavnaya"))


client = app.test_client()
print("Приложение и тестовый клиент готовы.")

## Eksperyment 1 - Strona główna (GET /)

In [ ]:
otvet = client.get("/")
telo = otvet.get_data(as_text=True)

print("Код ответа:", otvet.status_code)
print(telo)

## Sprawdzenie wyniku

In [ ]:
assert otvet.status_code == 200
assert "Выучить основы Python" in telo
assert "Собрать сайт на Flask" in telo
print("Верно: главная страница отдаёт список задач с кодом 200.")

## Eksperyment 2 - Dynamiczna trasa /privet/<imya>

In [ ]:
otvet2 = client.get("/privet/Ада")
telo2 = otvet2.get_data(as_text=True)

print(telo2)
assert "Привет, Ада!" in telo2
print("Верно: значение из адреса подставилось в шаблон.")

## Eksperyment 3 — wysyłanie formularza (POST /dobavit)

In [ ]:
kolichestvo_do = len(zadachi)

otvet3 = client.post("/dobavit", data={"zadacha": "Прочитать книгу"})
print("Код ответа:", otvet3.status_code)   # 302 — редирект на главную
print("Задач было:", kolichestvo_do, "-> стало:", len(zadachi))

assert otvet3.status_code == 302
assert len(zadachi) == kolichestvo_do + 1
assert "Прочитать книгу" in zadachi
print("Верно: POST-запрос добавил новую задачу и вернул редирект.")

## Zadanie niezależne od zadania ★★

Złóż formularz z pustym zadaniem (`{"zadacha": "   "}`) i upewnić się, że zadanie lista się nie zmieniło, tak jak teraz `app.py`.

In [ ]:
kolichestvo_do2 = len(zadachi)
client.post("/dobavit", data={"zadacha": "   "})

assert len(zadachi) == kolichestvo_do2
print("Верно: пустая (только пробелы) задача не была добавлена.")